## Задание 3. Загрузка «грязных» данных

Перед началом работы загрузите из личного кабинета файлы transactions_dirty.csv и transactions_dirty.json.

Файл общий для нескольких счетов: нужно загружать в конкретный счёт только его операции (фильтрация по номеру счёта / типу - по данным файла).

Правила
1. Реализуйте загрузку истории из CSV и\или из JSON.

2. Перед записью в историю отбрасывайте невалидные строки: опечатки в типе операции, пустые или отрицательные суммы, некорректные даты, некорректный статус и т.п.

3. Для сберегательного счёта допустимы типы операций deposit, withdraw, interest; для расчётного - только deposit и withdraw. Операции, недопустимые для типа счёта, невалидны.

4. После успешной загрузки валидных операций баланс счёта должен соответствовать последовательности принятых операций.

**Демонстрация**

Загрузите данные хотя бы из одного формата в расчётный и в сберегательный счёт, покажите обновлённый баланс и фрагмент истории.

**Дополнительное задание. Отчёт об отбраковке**

Помимо «молчаливого» отбрасывания мусора сформируйте явный результат очистки:

- сколько строк принято,

- сколько отклонено,

- почему отклонена каждая (или группа) строк - битая дата, пустая сумма, опечатка в типе (diposit), interest на расчётном счёте, неверный статус и т.д.

Формат отчёта на ваш выбор (список, датафрейм, печать сводки) - главное, чтобы его можно было показать в ноутбуке и по нему было понятно, что произошло с данными.

Этот пункт не обязателен, но без него сложно претендовать на верхнюю часть шкалы: он показывает зрелость работы с внешними данными, а не только «файл как-то прочитался».

In [1]:
import json
import math
from datetime import datetime
import pandas as pd

Сначала посмотрим на первые строки CSV-файла

In [2]:
with open("transactions_dirty.csv", encoding="utf-8") as f:
    for i in range(6):
        print(f.readline().strip())

account_number,account_type,date,operation,amount,balance_after,status
ACC-100001,checking,2025-09-27 22:17:26,deposit,921.0,2121.0,success
ACC-100001,checking,2025-09-27 22:17:26,deposit,607.0,2728.0,success
ACC-100001,checking,2025-09-28 22:17:26,deposit,488.0,3216.0,success
ACC-100001,checking,28/09/2025 22:17,deposit,129.0,3345.0,success
ACC-100001,checking,2025-09-29 22:17:26,deposit,880.0,4225.0,success


Теперь посмотрим на таблицу целиком: какие есть счета, какие значения встречаются в колонках.
Все значения читаем как текст.

In [3]:
df_csv = pd.read_csv("transactions_dirty.csv", dtype=str, keep_default_na=False)
print("Размер таблицы (строк, колонок):", df_csv.shape)
print("Колонки:", df_csv.columns.tolist())
print()
print("Какие счета есть в файле:")
print(df_csv.groupby(["account_number", "account_type"]).size())
print()
print("Какие значения в колонке operation:")
print(df_csv["operation"].value_counts())
print()
print("Какие значения в колонке status:")
print(df_csv["status"].value_counts())

Размер таблицы (строк, колонок): (105, 7)
Колонки: ['account_number', 'account_type', 'date', 'operation', 'amount', 'balance_after', 'status']

Какие счета есть в файле:
account_number  account_type
ACC-100001      checking        26
ACC-100002      savings         27
ACC-100003      checking        21
ACC-100004      savings         31
dtype: int64

Какие значения в колонке operation:
operation
deposit     52
withdraw    32
interest    14
diposit      3
             2
wdraw        2
Name: count, dtype: int64

Какие значения в колонке status:
status
success    98
succes      7
Name: count, dtype: int64


Уже видно мусор: опечатки и пустые значения.

## 3. Загрузка CSV и JSON в одну таблицу

Напишем функцию load_all_files. Что она делает:
1. читает CSV и JSON;
2. приводит всё к тексту, чтобы таблицы были одинаковыми (в JSON суммы - это числа, а пропуски - NaN);
3. добавляет две служебные колонки: source (откуда строка: csv или json) и record_no (номер записи в своём файле) - они нужны, чтобы в отчёте было понятно, где искать плохую строку;
4. склеивает таблицы;
5. помечает дубликаты - строки, которые полностью повторяют строку выше (колонка is_duplicate).

CSV и JSON содержат одни и те же операции. Если их просто склеить, каждая операция посчитается дважды и баланс будет неверным.

In [4]:
# Колонки, которые есть в обоих файлах
data_columns = ['account_number', 'account_type', 'date', 'operation',
                'amount', 'balance_after', 'status']


def load_all_files(csv_name, json_name):
    df_csv = pd.read_csv(csv_name, dtype=str, keep_default_na=False)
    
    with open(json_name, 'r', encoding='utf-8') as f:
        data_json = json.load(f)
    df_json = pd.DataFrame(data_json)
    df_json = df_json.astype(object).fillna('').astype(str)

   
    df_csv['source'] = 'csv'
    df_csv['record_no'] = df_csv.index + 1
    df_json['source'] = 'json'
    df_json['record_no'] = df_json.index + 1

   
    df_all = pd.concat([df_csv, df_json], ignore_index=True)

    df_all['is_duplicate'] = df_all.duplicated(subset=data_columns)

    return df_all


df_all = load_all_files('transactions_dirty.csv', 'transactions_dirty.json')
print('Всего строк в объединённой таблице:', len(df_all))

Всего строк в объединённой таблице: 210


Проверим, чем отличаются файлы между собой.

In [5]:
csv_part = df_all[df_all['source'] == 'csv'][data_columns].reset_index(drop=True)
json_part = df_all[df_all['source'] == 'json'][data_columns].reset_index(drop=True)

print('CSV и JSON полностью совпадают:', csv_part.equals(json_part))
print('Строк в CSV:', len(csv_part))
print('Строк в JSON:', len(json_part))
print('Строк-дубликатов в объединённой таблице:', df_all['is_duplicate'].sum())

CSV и JSON полностью совпадают: True
Строк в CSV: 105
Строк в JSON: 105
Строк-дубликатов в объединённой таблице: 110


**Вывод:** файлы одинаковые, поэтому вся копия из JSON будет помечена как дубликат. Оставляю в работе оба файла т.к в задании стоит И/ИЛИ в отношении того, какие файлы должны быть использованы. 

In [6]:
df_all.tail(10)

,account_number,account_type,date,operation,amount,balance_after,status,source,record_no,is_duplicate
200,ACC-100004,savings,2025-10-17 22:17:26,deposit,864.0,9393.95,success,json,96,True
201,ACC-100004,savings,2025-10-19 22:17:26,deposit,671.0,10064.95,success,json,97,True
202,ACC-100004,savings,2025-10-19 22:17:26,interest,503.25,10568.2,success,json,98,True
203,ACC-100004,savings,2025-10-19 22:17:26,,412.0,10156.2,success,json,99,True
204,ACC-100004,savings,2025-10-19 22:17:26,deposit,696.0,10852.2,success,json,100,True
205,ACC-100002,savings,09/10/2025 22:17,deposit,638.0,5241.27,success,json,101,True
206,ACC-100003,checking,2025-10-12 22:17:26,deposit,-115.0,3547.0,success,json,102,True
207,ACC-100004,savings,01/10/2025 22:17,withdraw,309.0,4247.0,success,json,103,True
208,ACC-100002,savings,2025-10-19 22:17:26,withdraw,300.0,6042.66,succes,json,104,True
209,ACC-100001,checking,2025-10-16 22:17:26,withdraw,393.0,7431.0,success,json,105,True


## 4. Проверка строк

Ошибки:
- дубликат строки
- неверный статус
- битая дата или дата в неподходящем формате
- пустая сумма, сумма не число, сумма меньше или равна нулю;
- пустой тип операции или опечатка в нём
- операция, которая не подходит типу счёта

Каждая проверка возвращает пару: True/False и причину.

In [7]:
# Проверка даты: в файлах встречаются два формата
def is_valid_date(date_str):
    possible_formats = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']

    for fmt in possible_formats:
        try:
            datetime.strptime(date_str, fmt)
            return True
        except ValueError:
            pass

    return False


# Общая проверка строки: дубликат, статус, дата, сумма
def check_basic_validity(row):
    if row['is_duplicate']:
        return False, 'Дубликат строки'

    status = row['status'].strip()
    if status != 'success':
        return False, 'Неверный статус'

    date_val = row['date'].strip()
    if not is_valid_date(date_val):
        return False, 'Битая дата'

    amount_text = row['amount'].strip()
    if amount_text == '':
        return False, 'Пустая сумма'

    try:
        amount = float(amount_text)
    except ValueError:
        return False, 'Сумма не число'

    if math.isnan(amount) or math.isinf(amount):
        return False, 'Сумма не число'

    if amount <= 0:
        return False, 'Сумма <= 0'

    return True, 'OK'


# Проверка: подходит ли тип операции для типа счёта
def check_operation_for_account(operation, account_type):
    op = operation.strip()

    if op == '':
        return False, 'Пустой тип операции'

    if account_type == 'checking':
        allowed_operations = ['deposit', 'withdraw']
    elif account_type == 'savings':
        allowed_operations = ['deposit', 'withdraw', 'interest']
    else:
        return False, 'Неизвестный тип счёта'

    if op not in allowed_operations:
        if op == 'interest':
            return False, 'interest на расчётном счёте'
        return False, 'Опечатка в типе операции'

    return True, 'OK'

## 5. Загрузка операций в счёт

Файл общий для нескольких счетов, поэтому функция process_account_transactions берёт только строки нужного счёта (по account_number). Тип счёта (checking или savings) она берёт из самого файла.

Для каждой строки:
1. проверяем строку 
2. если всё ОК - проверяем, подходит ли операция типу счёта;
3. если строка не прошла проверку - записываем её в таблицу отбраковки вместе с причиной
4. если прошла - меняем баланс

Начальный баланс. В файлах его нет, но есть колонка `balance_after` (баланс после операции). Если взять первую операцию счёта и вычесть её сумму из balance_after.

Саму колонку balance_after для проверки мы не используем: она посчитана и с учётом «грязных» строк, которые мы как раз отбрасываем.

In [8]:
# Начальные балансы счетов (посчитаны по первой операции каждого счёта)
initial_balances = {
    'ACC-100001': 1200.0,
    'ACC-100002': 1500.0,
    'ACC-100003': 800.0,
    'ACC-100004': 2000.0,
}

history_columns = ['Источник', 'Запись №', 'Дата', 'Операция', 'Сумма', 'Баланс после']
rejected_columns = ['Счёт', 'Источник', 'Запись №', 'Дата', 'Операция', 'Сумма', 'Статус', 'Причина']


def process_account_transactions(df, account_number, initial_balance):
    # Берём только строки нужного счёта
    account_rows = df[df['account_number'] == account_number]

    # Тип счёта берём из файла
    account_type = account_rows['account_type'].values[0]

    current_balance = initial_balance
    valid_history = []
    rejected_report = []

    for idx, row in account_rows.iterrows():
        # Сначала общая проверка, потом проверка типа операции
        ok, reason = check_basic_validity(row)
        if ok:
            ok, reason = check_operation_for_account(row['operation'], account_type)

        # Плохую строку записываем в отчёт и переходим к следующей
        if not ok:
            rejected_report.append({
                'Счёт': account_number,
                'Источник': row['source'],
                'Запись №': row['record_no'],
                'Дата': row['date'],
                'Операция': row['operation'],
                'Сумма': row['amount'],
                'Статус': row['status'],
                'Причина': reason,
            })
            continue

        # Хорошая строка: меняем баланс
        amount = float(row['amount'])
        op = row['operation'].strip()

        if op in ['deposit', 'interest']:
            current_balance += amount
        elif op == 'withdraw':
            current_balance -= amount

        valid_history.append({
            'Источник': row['source'],
            'Запись №': row['record_no'],
            'Дата': row['date'],
            'Операция': op,
            'Сумма': amount,
            'Баланс после': round(current_balance, 2),
        })

    history = pd.DataFrame(valid_history, columns=history_columns)
    rejected = pd.DataFrame(rejected_report, columns=rejected_columns)
    return account_type, current_balance, history, rejected

## 6. Проверка



### Расчётный счёт ACC-100001 (checking)

In [9]:
acc_type, bal_checking, hist_checking, rej_checking = process_account_transactions(
    df_all, 'ACC-100001', initial_balances['ACC-100001']
)

print('Счёт ACC-100001, тип:', acc_type)
print('Начальный баланс:', initial_balances['ACC-100001'])
print('Принято операций:', len(hist_checking))
print('Отклонено строк:', len(rej_checking))
print(f'Итоговый баланс: {bal_checking:.2f}')
print()
print('Первые 5 принятых операций:')
display(hist_checking.head(5))

Счёт ACC-100001, тип: checking
Начальный баланс: 1200.0
Принято операций: 17
Отклонено строк: 35
Итоговый баланс: 4864.00

Первые 5 принятых операций:


,Источник,Запись №,Дата,Операция,Сумма,Баланс после
0,csv,1,2025-09-27 22:17:26,deposit,921.0,2121.0
1,csv,2,2025-09-27 22:17:26,deposit,607.0,2728.0
2,csv,3,2025-09-28 22:17:26,deposit,488.0,3216.0
3,csv,4,28/09/2025 22:17,deposit,129.0,3345.0
4,csv,5,2025-09-29 22:17:26,deposit,880.0,4225.0


### Сберегательный счёт ACC-100002 (savings)

In [10]:
acc_type, bal_savings, hist_savings, rej_savings = process_account_transactions(
    df_all, 'ACC-100002', initial_balances['ACC-100002']
)

print('Счёт ACC-100002, тип:', acc_type)
print('Начальный баланс:', initial_balances['ACC-100002'])
print('Принято операций:', len(hist_savings))
print('Отклонено строк:', len(rej_savings))
print(f'Итоговый баланс: {bal_savings:.2f}')
print()
print('Первые 5 принятых операций:')
display(hist_savings.head(5))

Счёт ACC-100002, тип: savings
Начальный баланс: 1500.0
Принято операций: 18
Отклонено строк: 36
Итоговый баланс: 3991.24

Первые 5 принятых операций:


,Источник,Запись №,Дата,Операция,Сумма,Баланс после
0,csv,26,2025-09-27 22:17:26,deposit,469.00,1969.00
1,csv,27,2025-09-27 22:17:26,withdraw,392.00,1577.00
2,csv,28,2025-09-28 22:17:26,deposit,317.00,1894.00
3,csv,31,2025-10-02 22:17:26,deposit,840.00,2734.00
4,csv,32,04/10/2025 22:17,interest,183.28,2917.28


## 7. Дополнительное задание. Отчёт об отбраковке

### 7.1. Отчёт по двум демонстрационным счетам

Для каждого счёта покажем таблицу всех отклонённых строк и сводку: сколько строк отклонено по каждой причине.

In [11]:
print('Отклонённые строки счёта ACC-100001:')
display(rej_checking)

print()
print('Сколько строк отклонено по каждой причине (ACC-100001):')
print(rej_checking['Причина'].value_counts())

Отклонённые строки счёта ACC-100001:


,Счёт,Источник,Запись №,Дата,Операция,Сумма,Статус,Причина
0,ACC-100001,csv,6,2025-09-29 22:17:26,withdraw,,success,Пустая сумма
1,ACC-100001,csv,9,2025-10-01 22:17:26,,654.0,success,Пустой тип операции
2,ACC-100001,csv,11,2025-10-03 22:17:26,interest,532.0,success,interest на расчётном счёте
3,ACC-100001,csv,14,2025-17-34 12:00:00,deposit,793.0,success,Битая дата
4,ACC-100001,csv,16,2025-10-06 22:17:26,diposit,669.0,success,Опечатка в типе операции
5,ACC-100001,csv,17,2025-10-08 22:17:26,interest,744.0,success,interest на расчётном счёте
6,ACC-100001,csv,18,2023-16-40 12:00:00,withdraw,111.0,success,Битая дата
7,ACC-100001,csv,25,2025-10-19 22:17:26,deposit,916.0,succes,Неверный статус
8,ACC-100001,csv,105,2025-10-16 22:17:26,withdraw,393.0,success,Дубликат строки
9,ACC-100001,json,1,2025-09-27 22:17:26,deposit,921.0,success,Дубликат строки



Сколько строк отклонено по каждой причине (ACC-100001):
Причина
Дубликат строки                27
interest на расчётном счёте     2
Битая дата                      2
Пустая сумма                    1
Пустой тип операции             1
Опечатка в типе операции        1
Неверный статус                 1
Name: count, dtype: int64


In [12]:
print('Отклонённые строки счёта ACC-100002:')
display(rej_savings)

print()
print('Сколько строк отклонено по каждой причине (ACC-100002):')
print(rej_savings['Причина'].value_counts())

Отклонённые строки счёта ACC-100002:


,Счёт,Источник,Запись №,Дата,Операция,Сумма,Статус,Причина
0,ACC-100002,csv,29,2023-22-32 12:00:00,deposit,797.0,success,Битая дата
1,ACC-100002,csv,30,2025-09-30 22:17:26,interest,,success,Пустая сумма
2,ACC-100002,csv,34,2025-10-08 22:17:26,deposit,-136.0,success,Сумма <= 0
3,ACC-100002,csv,38,2025-10-09 22:17:26,diposit,432.0,success,Опечатка в типе операции
4,ACC-100002,csv,45,2025-10-16 22:17:26,deposit,-872.0,success,Сумма <= 0
5,ACC-100002,csv,47,2025-10-19 22:17:26,withdraw,300.0,succes,Неверный статус
6,ACC-100002,csv,49,2025-10-22 22:17:26,wdraw,457.0,success,Опечатка в типе операции
7,ACC-100002,csv,101,09/10/2025 22:17,deposit,638.0,success,Дубликат строки
8,ACC-100002,csv,104,2025-10-19 22:17:26,withdraw,300.0,succes,Дубликат строки
9,ACC-100002,json,26,2025-09-27 22:17:26,deposit,469.0,success,Дубликат строки



Сколько строк отклонено по каждой причине (ACC-100002):
Причина
Дубликат строки             29
Сумма <= 0                   2
Опечатка в типе операции     2
Битая дата                   1
Пустая сумма                 1
Неверный статус              1
Name: count, dtype: int64


### 7.2. Сводка по всем счетам

In [13]:
summary = []
all_rejected = []

for number in initial_balances:
    acc_type, balance, history, rejected = process_account_transactions(
        df_all, number, initial_balances[number]
    )

    rows_total = len(df_all[df_all['account_number'] == number])

    summary.append({
        'Счёт': number,
        'Тип': acc_type,
        'Строк в файлах': rows_total,
        'Принято': len(history),
        'Отклонено': len(rejected),
        'Начальный баланс': initial_balances[number],
        'Итоговый баланс': round(balance, 2),
    })
    all_rejected.append(rejected)

summary_table = pd.DataFrame(summary)
display(summary_table)

total_checked = summary_table['Принято'].sum() + summary_table['Отклонено'].sum()
print('Всего строк в таблице:', len(df_all))
print('Принято + отклонено:', total_checked)
print('Ничего не потерялось:', total_checked == len(df_all))

,Счёт,Тип,Строк в файлах,Принято,Отклонено,Начальный баланс,Итоговый баланс
0,ACC-100001,checking,52,17,35,1200.0,4864.00
1,ACC-100002,savings,54,18,36,1500.0,3991.24
2,ACC-100003,checking,42,13,29,800.0,5446.00
3,ACC-100004,savings,62,19,43,2000.0,10057.25


Всего строк в таблице: 210
Принято + отклонено: 210
Ничего не потерялось: True


### 7.3. Причины отбраковки: все счета вместе

In [14]:
all_rejected_table = pd.concat(all_rejected, ignore_index=True)

print('Причины отбраковки (все счета):')
print(all_rejected_table['Причина'].value_counts())

print()
print('Причина по счетам:')
display(pd.crosstab(all_rejected_table['Причина'], all_rejected_table['Счёт']))

Причины отбраковки (все счета):
Причина
Дубликат строки                110
interest на расчётном счёте      6
Битая дата                       6
Неверный статус                  6
Опечатка в типе операции         5
Сумма <= 0                       5
Пустая сумма                     3
Пустой тип операции              2
Name: count, dtype: int64

Причина по счетам:


Счёт,ACC-100001,ACC-100002,ACC-100003,ACC-100004
Причина,,,,
interest на расчётном счёте,2,0,4,0
Битая дата,2,1,1,2
Дубликат строки,27,29,22,32
Неверный статус,1,1,0,4
Опечатка в типе операции,1,2,1,1
Пустая сумма,1,1,0,1
Пустой тип операции,1,0,0,1
Сумма <= 0,0,2,1,2


## Вывод

- CSV и JSON содержат одни и те же операции, поэтому копии мы не принимаем: строка, которая уже встречалась, отклоняется как «Дубликат строки». Иначе все операции были бы учтены дважды.
- Остальной мусор тоже отброшен и объяснён в отчёте: неверный статус, битая дата, пустая или отрицательная сумма, пустой тип и опечатки в типе операции.
- Баланс каждого счёта считается только по принятым операциям и проверяется вторым способом (пересчёт по таблице истории).
- В отчёте у каждой отклонённой строки указан файл (Источник) и номер записи в нём (Запись №), так что плохую строку легко найти.